# 코드 수정을 반복하는 데이터 분석 Agent

코드를 **생성 → 실제 실행 → 에러가 나면 스스로 수정** 하는 루프를 만든다. Reflection 패턴을 "코드" 에 적용한 것 — 평가자가 사람이 아니라 **파이썬 인터프리터(실행 성공/실패)** 다.

예제: CSV 데이터셋을 받아 전처리 코드를 작성하고, 실행해보며 에러를 잡아 고친다.

```
START → chatbot ──(도구 필요?)──▶ add_context → generate → code_check ──(성공?)──▶ END
          │ (아니면)                                          │ (실패)
          ▼                                                   ▼
         END                                       reflect → code_check (재검사)
```

> OpenAI(구조화/도구) + Anthropic(코드 생성) 두 LLM 을 함께 쓴다.

## 환경 변수 준비

프로젝트 루트의 `.env` 에 키를 넣고 `load_dotenv()` 로 불러온다.

```
OPENAI_API_KEY=sk-...
TAVILY_API_KEY=tvly-...     # 웹검색이 필요한 노트북
ANTHROPIC_API_KEY=sk-ant-... # Claude 를 쓰는 노트북
```

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY 가 .env 에 없습니다"
assert os.environ.get("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY 가 .env 에 없습니다"
print("환경변수 로드 완료:", "OPENAI_API_KEY", "ANTHROPIC_API_KEY")

## Step 1. 데이터 통계를 주는 도구

[basics 복습] `@tool` 로 CSV 의 `describe()` 통계를 반환하는 도구를 만든다. 이 통계가 전처리 코드 작성의 컨텍스트가 된다.

In [ ]:
from langchain_core.tools import tool
import pandas as pd

@tool
def describe_data(csv: str) -> str:
    """데이터프레임의 통계 요약을 반환한다.

    Args:
        csv: CSV 파일 경로 또는 URL
    """
    df = pd.read_csv(csv)
    return f"Data: {csv}\n" + df.describe(include="all").to_string()

tools = [describe_data]

## Step 2. State 정의
[basics 복습] `MessagesState` 를 상속해 제어용 필드(error/context/generation/iterations)를 추가한다.

In [ ]:
from langgraph.graph import StateGraph, MessagesState

class State(MessagesState):
    error: str        # 'yes' / 'no' — 코드 실행 에러 여부
    context: str      # 데이터 통계 요약
    generation: object  # 생성된 코드 솔루션
    iterations: int   # 재시도 횟수

graph_builder = StateGraph(State)

## Step 3. LLM 준비 + 코드 출력 스키마

코드 생성은 Claude, 도구 호출/구조화는 GPT 를 쓴다. [basics 복습] 코드 솔루션을 `prefix / imports / code` 로 나눈 구조화 스키마로 받는다.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic
from pydantic import BaseModel, Field

llm_gpt = ChatOpenAI(model="gpt-4o-mini")
llm_claude = ChatAnthropic(model="claude-3-7-sonnet-20250219")

class Code(BaseModel):
    """코드 솔루션 스키마"""
    prefix: str = Field(description="문제와 접근 방식 설명")
    imports: str = Field(description="import 문")
    code: str = Field(description="import 를 제외한 코드 본문")

## Step 4. 노드들 정의

### 4-1. chatbot — 도구 호출 여부 판단
[basics 복습] `bind_tools` 한 LLM 이 데이터 통계 도구를 부를지 결정한다.

In [ ]:
llm_with_tools = llm_gpt.bind_tools(tools=[describe_data])

def chatbot(state: State):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

graph_builder.add_node("chatbot", chatbot)

### 4-2. add_context — 도구 실행으로 데이터 통계 확보
[basics 복습] 마지막 AIMessage 의 `tool_calls` 를 읽어 도구를 직접 실행한다.

In [ ]:
def add_context(state: State):
    message = state["messages"][-1]
    describe_str = ""
    for tool_call in message.tool_calls:
        for t in tools:
            if t.name == tool_call["name"]:
                describe_str = t.invoke(tool_call["args"])
    return {"context": describe_str}

graph_builder.add_node("add_context", add_context)

[basics 복습] 조건부 라우팅 — 도구 호출이 있으면 `add_context`, 없으면 종료.

In [ ]:
from langgraph.graph import END

def guardrail_route(state: State):
    messages = state.get("messages", [])
    if not messages:
        raise ValueError("No messages found")
    ai_message = messages[-1]
    if hasattr(ai_message, "tool_calls") and len(ai_message.tool_calls) > 0:
        return "add_context"
    return END

graph_builder.add_conditional_edges(
    "chatbot", guardrail_route, {"add_context": "add_context", END: END}
)

### 4-3. generate — 전처리 코드 생성
Claude 가 코드를 생성하고, GPT 가 `Code` 스키마로 구조화한다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

GENERATE_CODE_TEMPLATE = """
Given the following pandas `describe()` output of a dataset,
write a directly executable Python code to:
1. handle missing values, 2. convert categorical columns,
3. any additional preprocessing, 4. prepare for machine learning.

Describe result:
 ------- 
 {context} 
 ------- 

Do not wrap the code in a function or backticks. Write a flat script that runs immediately.
Provide a description, then imports, then the functioning code block.
"""
code_gen_prompt = ChatPromptTemplate.from_messages([("user", GENERATE_CODE_TEMPLATE)])

def generate(state: State):
    print("##### GENERATING CODE #####")
    generated = llm_claude.invoke(code_gen_prompt.format_messages(context=state["context"]))
    code_solution = llm_gpt.with_structured_output(Code).invoke(generated.content)
    messages = [("assistant",
        f"{code_solution.prefix}\nImports: {code_solution.imports}\nCode: {code_solution.code}")]
    return {"generation": code_solution, "messages": messages}

graph_builder.add_node("generate", generate)

### 4-4. code_check — 코드를 실제 실행해 검증
import 와 코드 본문을 `exec` 로 실제 실행한다. 실패하면 에러 메시지를 state 에 담고 `error='yes'`.

In [ ]:
def code_check(state: State):
    print("##### CHECKING CODE #####")
    code_solution = state["generation"]
    imports, code_body = code_solution.imports, code_solution.code

    # import 검사
    try:
        exec(imports)
    except Exception as e:
        return {"generation": code_solution, "error": "yes",
                "messages": [("user", f"import 테스트 실패: {e}")]}
    # 실행 검사
    try:
        exec(imports + "\n" + code_body)
    except Exception as e:
        return {"generation": code_solution, "error": "yes",
                "messages": [("user", f"코드 실행 테스트 실패: {e}")]}

    print("---NO ERRORS---")
    return {"generation": code_solution, "error": "no"}

graph_builder.add_node("code_check", code_check)

### 4-5. reflect — 에러를 보고 코드 수정
실행 에러 메시지와 원본 코드를 주고 Claude 가 수정본을 만든다.

In [ ]:
reflect_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are given an error from a Python script and the original code. "
     "Provide corrected code that runs without errors and keeps the intended functionality."),
    ("user",
     "--- ERROR ---\n{error}\n--- ORIGINAL CODE ---\n{code_solution}\n\n"
     "Provide a description, then imports, then the functioning code block."),
])

def reflect(state: State):
    print("---REFLECTING---")
    error = state["messages"][-1].content
    cs = state["generation"]
    code_str = f"{cs.prefix}\nImports: {cs.imports}\nCode: {cs.code}"
    corrected = llm_claude.invoke(reflect_prompt.format_messages(error=error, code_solution=code_str))
    fixed = llm_gpt.with_structured_output(Code).invoke(corrected.content)
    messages = [("assistant",
        f"{fixed.prefix}\nImports: {fixed.imports}\nCode: {fixed.code}")]
    return {"generation": fixed, "messages": messages, "iterations": state["iterations"] + 1}

graph_builder.add_node("reflect", reflect)

## Step 5. 그래프 연결 + 컴파일
에러가 없거나 최대 반복(5회)에 도달하면 종료, 아니면 reflect 로 보내 재검사 루프.

In [ ]:
from langgraph.graph import START, END

max_iterations = 5

def decide_to_finish(state: State):
    if state["error"] == "no" or state["iterations"] == max_iterations:
        print("---DECISION: FINISH---")
        return "end"
    print("---DECISION: RE-TRY---")
    return "reflect"

graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("add_context", "generate")
graph_builder.add_edge("generate", "code_check")
graph_builder.add_conditional_edges(
    "code_check", decide_to_finish, {"end": END, "reflect": "reflect"}
)
graph_builder.add_edge("reflect", "code_check")
graph = graph_builder.compile()

In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph(xray=True).draw_mermaid_png()))
except Exception:
    print(graph.get_graph().draw_mermaid())

## 실행
공개 타이타닉 데이터셋 URL 로 전처리 코드를 생성·검증한다.

In [ ]:
csv_url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"
question = f"{csv_url} 데이터의 전처리를 부탁합니다!"

solution = graph.invoke({"messages": [("user", question)], "iterations": 0, "error": ""})

print(solution["generation"].prefix, "\n")
print(solution["generation"].imports, "\n")
print(solution["generation"].code)

## 정리

- 코드 생성 → **실제 실행으로 검증** → 에러 시 수정의 반복 (평가자 = 인터프리터)
- `code_check` 노드가 `exec` 로 import·실행을 검사해 `error` 플래그를 세움
- `decide_to_finish` 가 에러 여부·반복 횟수로 종료/재시도를 결정
- 코드 생성(Claude) + 구조화/도구(GPT) 처럼 **여러 LLM 을 역할 분담** 할 수 있다

이로써 '반복하고 수정하는 Agent' 패턴(Reflection · Plan&Execute · 코드수정)을 모두 다뤘다.